# Phase 3 — Decomposed multi-head model

Implements the architecture from `fpl_project_context.md` (Phase 3):

- Shared backbone: **128 → 64 → 32** (ReLU, batch norm, dropout)
- Heads: **P(play)** (`minutes > 0`), **P(60+)** (`minutes ≥ 60`), **P(goal | played)**, **P(assist | played)**, **P(clean sheet)**, **E[bonus]**, **E[goals conceded]** (DEF/GK)
- Appearance: `P(play) × (1 + P(60+))` → 0 if benched, 1 if sub, 2 if 60+ (FPL rules)
- **Recombine** head outputs into expected FPL points using official scoring rules by position

Compared against baselines saved in `results/phase3_model_comparison.csv` from notebook `02`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "fpl_model_dataset.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
BASELINE_METRICS_PATH = RESULTS_DIR / "phase3_model_comparison.csv"
BASELINE_VAL_PREDS_PATH = RESULTS_DIR / "val_2024_25_predictions.csv"

TRAIN_SEASONS = [
    "2016-17", "2017-18", "2018-19", "2019-20",
    "2020-21", "2021-22", "2022-23", "2023-24",
]
VAL_SEASON = "2024-25"
TARGET = "total_points"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

Project root: C:\FPL_project
Device: cpu


In [2]:
NUMERIC_FEATURES = [
    "total_points_roll3", "total_points_roll5",
    "minutes_roll3", "minutes_roll5",
    "goals_scored_roll3", "goals_scored_roll5",
    "assists_roll3", "assists_roll5",
    "expected_goals_roll3", "expected_goals_roll5",
    "expected_assists_roll3", "expected_assists_roll5",
    "team_goals_scored_gw_roll5", "team_goals_conceded_gw_roll5", "team_points_gw_roll5",
    "opponent_team_points_roll5", "opponent_team_gc_roll5",
    "last_season_ppg", "last_season_minutes_share",
    "was_home", "rest_days",
    "value", "selected", "transfers_in", "transfers_out", "transfers_balance",
]

POSITION_MAP = {"GK": 1, "GKP": 1, "DEF": 2, "MID": 3, "AM": 3, "FWD": 4}


def load_modeling_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["position_id"] = df["position"].map(POSITION_MAP).fillna(3).astype(int)
    df["is_promoted_team"] = df["is_promoted_team"].map({True: 1.0, False: 0.0}).fillna(0.0)
    df["was_home"] = pd.to_numeric(df["was_home"], errors="coerce").fillna(0.0)
    return df


def build_feature_matrix(df: pd.DataFrame) -> np.ndarray:
    x_num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
    x_pos = df[["position_id"]].to_numpy(dtype=np.float32)
    x_prom = df[["is_promoted_team"]].to_numpy(dtype=np.float32)
    return np.hstack([x_num, x_pos, x_prom])


def split_sets(df: pd.DataFrame):
    train = df[df["season"].isin(TRAIN_SEASONS)].copy()
    val = df[df["season"] == VAL_SEASON].copy()
    return train, val


def scored_mask(df: pd.DataFrame) -> np.ndarray:
    return df["minutes"].fillna(0).to_numpy() > 0


def tensor_to_numpy(t: torch.Tensor) -> np.ndarray:
    # Workaround for NumPy 2.x + older PyTorch builds where .numpy() can fail.
    return np.asarray(t.detach().cpu().tolist(), dtype=np.float32)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "spearman": spearmanr(y_true, y_pred).statistic,
    }


def build_head_targets(df: pd.DataFrame) -> dict[str, np.ndarray]:
    # Sub-event labels for multi-task heads (same GW outcomes, not features).
    minutes = df["minutes"].fillna(0).to_numpy()
    starts = pd.to_numeric(df["starts"], errors="coerce")
    y_play = (minutes > 0).astype(np.float32)
    y_sixty = np.where(
        starts == 1,
        1.0,
        np.where(starts == 0, 0.0, (minutes >= 60).astype(float)),
    ).astype(np.float32)

    goals = pd.to_numeric(df["goals_scored"], errors="coerce").fillna(0).to_numpy()
    assists = pd.to_numeric(df["assists"], errors="coerce").fillna(0).to_numpy()
    cs = pd.to_numeric(df["clean_sheets"], errors="coerce").fillna(0).to_numpy()
    bonus = pd.to_numeric(df["bonus"], errors="coerce").fillna(0).to_numpy()
    gc = pd.to_numeric(df["goals_conceded"], errors="coerce").fillna(0).to_numpy()

    return {
        "play": y_play,
        "sixty": y_sixty,
        "goal": (goals > 0).astype(np.float32),
        "assist": (assists > 0).astype(np.float32),
        "cs": (cs > 0).astype(np.float32),
        "bonus": bonus.astype(np.float32),
        "gc": gc.astype(np.float32),
        "total_points": df[TARGET].to_numpy(dtype=np.float32),
        "position_id": df["position_id"].to_numpy(dtype=np.int64),
    }


df = load_modeling_table(DATA_PATH)
train_df, val_df = split_sets(df)
print(f"Train {len(train_df):,} | Val {len(val_df):,}")

Train 196,538 | Val 27,605


## FPL scoring recombination

Maps head outputs → expected points using position-specific rules (appearance, goals, assists, CS, conceded, bonus).

In [3]:
def expected_fpl_points(
    p_play: np.ndarray,
    p_sixty: np.ndarray,
    p_goal: np.ndarray,
    p_assist: np.ndarray,
    p_cs: np.ndarray,
    e_bonus: np.ndarray,
    e_gc: np.ndarray,
    position_id: np.ndarray,
) -> np.ndarray:
    """Combine decomposed head predictions into expected total_points."""
    pos = position_id.astype(int)
    goal_pts = np.select(
        [pos == 1, pos == 2, pos == 3, pos == 4],
        [6.0, 6.0, 5.0, 4.0],
        default=5.0,
    )
    cs_pts = np.select([pos <= 2, pos == 3], [4.0, 1.0], default=0.0)

    # FPL appearance: 0 if no play; 1 if 1-59 min; 2 if 60+.
    # E[pts | play] = P(60+) * 2 + (1 - P(60+)) * 1 = 1 + P(60+)
    appearance = p_play * (1.0 + p_sixty)

    # Attacking returns only if player actually plays (goals/assists when 60+ weighted higher in training).
    goal_points = p_play * p_sixty * p_goal * goal_pts
    assist_points = p_play * p_sixty * p_assist * 3.0
    cs_points = p_play * p_cs * cs_pts
    gc_penalty = p_play * np.where(pos <= 2, -0.5 * np.clip(e_gc, 0.0, None), 0.0)
    bonus_points = p_play * e_bonus

    return (appearance + goal_points + assist_points + cs_points + gc_penalty + bonus_points).astype(np.float32)


GOAL_PTS_LUT = torch.tensor([5.0, 6.0, 6.0, 5.0, 4.0])
CS_PTS_LUT = torch.tensor([0.0, 4.0, 4.0, 1.0, 0.0])


def expected_fpl_points_torch(out: dict, position_id: torch.Tensor) -> torch.Tensor:
    pos = position_id.long().clamp(0, 4)
    goal_pts = GOAL_PTS_LUT.to(position_id.device)[pos]
    cs_pts = CS_PTS_LUT.to(position_id.device)[pos]
    p_play, p_sixty = out["play"], out["sixty"]
    appearance = p_play * (1.0 + p_sixty)
    goal_points = p_play * p_sixty * out["goal"] * goal_pts
    assist_points = p_play * p_sixty * out["assist"] * 3.0
    cs_points = p_play * out["cs"] * cs_pts
    gc_penalty = p_play * torch.where(pos <= 2, -0.5 * out["gc"], torch.zeros_like(out["gc"]))
    bonus_points = p_play * out["bonus"]
    return appearance + goal_points + assist_points + cs_points + gc_penalty + bonus_points


# Sanity check
pos = np.array([3])
print("Bench (no play):", expected_fpl_points(np.array([0.0]), np.array([0.0]), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), pos))
print("Sub blank:", expected_fpl_points(np.array([1.0]), np.array([0.0]), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), pos))
print("Starter haul:", expected_fpl_points(np.array([1.0]), np.array([1.0]), np.array([0.4]), np.array([0.2]), np.array([0.0]), np.array([2.0]), np.array([0.0]), pos))

Bench (no play): [0.]
Sub blank: [1.]
Starter haul: [6.6]


## Decomposed model + training

In [11]:
class DecomposedFPLNet(nn.Module):
    def __init__(self, in_dim: int, dropout: float = 0.25):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head_play = nn.Linear(32, 1)
        self.head_sixty = nn.Linear(32, 1)
        self.head_goal = nn.Linear(32, 1)
        self.head_assist = nn.Linear(32, 1)
        self.head_cs = nn.Linear(32, 1)
        self.head_bonus = nn.Linear(32, 1)
        self.head_gc = nn.Linear(32, 1)

    def forward(self, x):
        h = self.backbone(x)
        return {
            "play": torch.sigmoid(self.head_play(h)).squeeze(-1),
            "sixty": torch.sigmoid(self.head_sixty(h)).squeeze(-1),
            "goal": torch.sigmoid(self.head_goal(h)).squeeze(-1),
            "assist": torch.sigmoid(self.head_assist(h)).squeeze(-1),
            "cs": torch.sigmoid(self.head_cs(h)).squeeze(-1),
            "bonus": F.relu(self.head_bonus(h)).squeeze(-1),
            "gc": F.relu(self.head_gc(h)).squeeze(-1),
        }


def heads_to_numpy(preds: dict) -> dict[str, np.ndarray]:
    return {k: tensor_to_numpy(v) for k, v in preds.items()}


def train_decomposed(
    X_train_s: np.ndarray,
    y_train: dict,
    X_val_s: np.ndarray,
    y_val: dict,
    val_played_mask: np.ndarray,
    epochs: int = 1000,
    batch_size: int = 4096,
    lr: float = 1e-3,
    patience: int = 6,
    aux_points_weight: float = 0.3,
):
    in_dim = X_train_s.shape[1]
    model = DecomposedFPLNet(in_dim).to(DEVICE)

    pos_train = torch.tensor(y_train["position_id"], dtype=torch.long)
    pos_val = y_val["position_id"]

    train_ds = TensorDataset(
        torch.tensor(X_train_s, dtype=torch.float32),
        torch.tensor(y_train["play"], dtype=torch.float32),
        torch.tensor(y_train["sixty"], dtype=torch.float32),
        torch.tensor(y_train["goal"], dtype=torch.float32),
        torch.tensor(y_train["assist"], dtype=torch.float32),
        torch.tensor(y_train["cs"], dtype=torch.float32),
        torch.tensor(y_train["bonus"], dtype=torch.float32),
        torch.tensor(y_train["gc"], dtype=torch.float32),
        torch.tensor(y_train["total_points"], dtype=torch.float32),
        pos_train,
    )
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_val_t = torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
    y_pts_val = y_val["total_points"]
    val_idx = np.where(val_played_mask)[0]

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    best_state = None
    best_val_mae = float("inf")
    stale = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, y_play, y_sixty, y_goal, y_assist, y_cs, y_bonus, y_gc, y_pts, pos_b in loader:
            xb = xb.to(DEVICE)
            y_play, y_sixty = y_play.to(DEVICE), y_sixty.to(DEVICE)
            y_goal, y_assist = y_goal.to(DEVICE), y_assist.to(DEVICE)
            y_cs, y_bonus, y_gc, y_pts = y_cs.to(DEVICE), y_bonus.to(DEVICE), y_gc.to(DEVICE), y_pts.to(DEVICE)
            pos_b = pos_b.to(DEVICE)
            out = model(xb)

            loss_play = F.binary_cross_entropy(out["play"], y_play)
            loss_sixty = F.binary_cross_entropy(out["sixty"], y_sixty, reduction="none")
            loss_sixty = (loss_sixty * (0.15 + 0.85 * y_play)).mean()
            goal_loss = F.binary_cross_entropy(out["goal"], y_goal, reduction="none")
            assist_loss = F.binary_cross_entropy(out["assist"], y_assist, reduction="none")
            w_sixty = y_play * (0.15 + 0.85 * y_sixty)
            loss_goal = (goal_loss * (0.15 + 0.85 * w_sixty)).mean()
            loss_assist = (assist_loss * (0.15 + 0.85 * w_sixty)).mean()

            cs_loss = F.binary_cross_entropy(out["cs"], y_cs, reduction="none")
            cs_mask = (pos_b <= 2) | (pos_b == 3)
            loss_cs = cs_loss[cs_mask].mean() if cs_mask.any() else cs_loss.mean()

            loss_bonus = F.mse_loss(out["bonus"], y_bonus)
            gc_loss = F.mse_loss(out["gc"], y_gc, reduction="none")
            def_gk = pos_b <= 2
            loss_gc = gc_loss[def_gk].mean() if def_gk.any() else torch.tensor(0.0, device=DEVICE)

            pred_pts = expected_fpl_points_torch(out, pos_b)
            loss_aux = F.mse_loss(pred_pts, y_pts)

            loss = (
                loss_play + loss_sixty + loss_goal + loss_assist + loss_cs
                + loss_bonus + 0.25 * loss_gc + aux_points_weight * loss_aux
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            out_val = model(X_val_t)
            pos_val_t = torch.as_tensor(pos_val.tolist(), dtype=torch.long, device=DEVICE)
            pred_val = tensor_to_numpy(expected_fpl_points_torch(out_val, pos_val_t))
            val_mae = mean_absolute_error(y_pts_val[val_idx], pred_val[val_idx])

        scheduler.step(val_mae)
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out_val = model(X_val_t)
        pos_val_t = torch.as_tensor(pos_val.tolist(), dtype=torch.long, device=DEVICE)
        pred_val = tensor_to_numpy(expected_fpl_points_torch(out_val, pos_val_t))
    return model, pred_val, out_val


X_train = build_feature_matrix(train_df)
X_val = build_feature_matrix(val_df)
y_train = build_head_targets(train_df)
y_val = build_head_targets(val_df)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

val_mask = scored_mask(val_df)
decomp_model, val_pred_decomposed, val_heads = train_decomposed(
    X_train_s, y_train, X_val_s, y_val, val_mask
)

decomp_metrics = regression_metrics(y_val["total_points"][val_mask], val_pred_decomposed[val_mask])
decomp_metrics["model"] = "mlp_decomposed"
pd.Series(decomp_metrics)

mae               1.847157
rmse              2.816642
r2                0.053053
spearman          0.367768
model       mlp_decomposed
dtype: object

## Compare to baselines (notebook 02)

In [14]:
if BASELINE_METRICS_PATH.exists():
    baseline_df = pd.read_csv(BASELINE_METRICS_PATH)
else:
    baseline_df = pd.DataFrame()
    print("Run 02_phase3_baseline_model.ipynb first to create", BASELINE_METRICS_PATH)

comparison = pd.concat([baseline_df, pd.DataFrame([decomp_metrics])], ignore_index=True)
comparison = comparison[["model", "mae", "rmse", "r2", "spearman"]].sort_values("mae")
comparison

,model,mae,rmse,r2,spearman
0,mlp_points,1.845740,2.861426,0.022702,0.358561
4,mlp_decomposed,1.847157,2.816642,0.053053,0.367768
1,ridge_linear,1.870430,2.840238,0.037121,0.347047
2,roll5_points,2.081290,3.067166,-0.122889,0.290270
3,last_season_ppg,2.188108,3.063853,-0.120464,0.226227


In [6]:
val_preds = val_df[["season", "element", "gw", "player_id", "name", "minutes", TARGET]].copy()
val_preds["pred_decomposed"] = val_pred_decomposed

if BASELINE_VAL_PREDS_PATH.exists():
    base_preds = pd.read_csv(BASELINE_VAL_PREDS_PATH)
    merge_keys = ["season", "element", "gw", "player_id"]
    val_preds = val_preds.merge(
        base_preds[merge_keys + ["pred_mlp", "pred_ridge"]],
        on=merge_keys,
        how="left",
    )
    played = val_preds[val_preds["minutes"].fillna(0) > 0].copy()
    print("Head-to-head on played rows (2024-25):")
    print("  MLP MAE:", mean_absolute_error(played[TARGET], played["pred_mlp"]))
    print("  Decomposed MAE:", mean_absolute_error(played[TARGET], played["pred_decomposed"]))
    print("  Spearman MLP:", spearmanr(played[TARGET], played["pred_mlp"]).statistic)
    print("  Spearman Decomposed:", spearmanr(played[TARGET], played["pred_decomposed"]).statistic)

Head-to-head on played rows (2024-25):
  MLP MAE: 1.8546730381268874
  Decomposed MAE: 1.864479660987854
  Spearman MLP: 0.36074956246809875
  Spearman Decomposed: 0.36631701106312553


In [7]:
# Example head outputs for top predicted hauls on validation.
with torch.no_grad():
    heads_np = heads_to_numpy(val_heads)

inspect = val_df[["name", "position", "gw", "minutes", TARGET]].copy()
prob_heads = ["play", "sixty", "goal", "assist", "cs"]
for k in prob_heads:
    inspect[f"p_{k}"] = heads_np[k]
inspect["e_bonus"] = heads_np["bonus"]
inspect["e_gc"] = heads_np["gc"]
inspect["pred_decomposed"] = val_pred_decomposed
inspect.loc[scored_mask(val_df)].nlargest(8, "pred_decomposed")

,name,position,gw,minutes,total_points,p_play,p_sixty,p_goal,p_assist,p_cs,e_bonus,e_gc,pred_decomposed
208346,Trent Alexander-Arnold,DEF,28,88,2,0.968856,0.902642,0.374339,0.300270,0.427366,0.541278,0.707257,6.433422
208993,Mohamed Salah,MID,30,90,3,0.999266,0.989543,0.350382,0.296686,0.458354,1.082885,1.345676,6.140611
208345,Trent Alexander-Arnold,DEF,27,76,6,0.958172,0.886664,0.362515,0.290071,0.418447,0.487569,0.683333,6.138544
208980,Mohamed Salah,MID,18,90,9,0.999593,0.992765,0.347702,0.269560,0.448351,1.165210,1.498971,6.132591
208997,Mohamed Salah,MID,34,90,8,0.998765,0.985529,0.356789,0.307908,0.463291,1.013474,1.242667,6.123205
208982,Mohamed Salah,MID,20,90,7,0.999539,0.992194,0.347672,0.273100,0.449822,1.143688,1.466644,6.120577
208978,Mohamed Salah,MID,16,90,5,0.999217,0.989048,0.349344,0.297670,0.445515,1.070385,1.365578,6.110976
208992,Mohamed Salah,MID,28,90,15,0.998678,0.984874,0.353397,0.315972,0.456513,0.996499,1.231966,6.103642


In [8]:
comparison.to_csv(RESULTS_DIR / "phase3_model_comparison_with_decomposed.csv", index=False)
val_preds.to_csv(RESULTS_DIR / "val_2024_25_predictions_with_decomposed.csv", index=False)
torch.save(
    {
        "model_state": decomp_model.state_dict(),
        "in_dim": X_train_s.shape[1],
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
    },
    RESULTS_DIR / "phase3_decomposed_mlp.pt",
)
print("Saved comparison + predictions + weights to", RESULTS_DIR)

Saved comparison + predictions + weights to C:\FPL_project\results
